# 01 Exploración de los datos

**Proyecto:** Análisis de estrategias de inversión en el S&P 500

En el notebook anterior (00_descarga_datos) descargué los datos que voy a usar en el proyecto. Antes de empezar a hacer cálculos, en este notebook quiero conocer bien esos datos y comprobar que no tienen errores, para que de esta forma los resultados sean lo más fiables posible.

### Datos que uso

Los tres archivos están en la carpeta Data/Raw:

**empresas.csv** tiene las 503 empresas del S&P 500, con su sector y su industria.

**precios.csv** tiene los precios de cada empresa día a día desde enero de 2016 hasta septiembre de 2026.

**indice.csv** tiene los precios de SPY, un fondo que copia al S&P 500 y que usaré para comparar.

### Qué voy a revisar

Voy a mirar si falta información, si hay datos repetidos o valores que no tienen sentido, y qué empresas tienen suficientes años de datos para poder compararlas entre ellas.

En este notebook no cambio ningún dato. Si encuentro algún problema, lo apunto para corregirlo después en el notebook de limpieza.

## 1. Cargar los datos

Cargo los tres archivos que generé en el notebook anterior.

El código detecta si se está ejecutando en Google Colab o en un ordenador. En Colab usa mi carpeta de Drive, y fuera de Colab usa la carpeta del proyecto descargado, para que cualquiera pueda ejecutarlo sin tener acceso a mi Drive.

In [4]:
# Importo pandas para trabajar con tablas
import pandas as pd

# Compruebo dónde se está ejecutando el notebook
try:
    # Si estoy en Colab, conecto mi Drive y uso mi carpeta
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_PROYECTO = "/content/drive/MyDrive/PORTFOLIO MARCOS GUERRERO SANCHEZ/analisis-estrategias-inversion/"
except ImportError:
    # Si no estoy en Colab, uso la carpeta donde esté el proyecto descargado
    # El ../ sube un nivel, porque el notebook está dentro de Notebooks/
    RUTA_PROYECTO = "../"

# Carpeta con los datos originales
RUTA = RUTA_PROYECTO + "Data/Raw/"

# Cargo los tres archivos
# Con el parámetro parse_dates la columna Date se lee como fecha y no como texto
empresas = pd.read_csv(RUTA + "empresas.csv")
precios = pd.read_csv(RUTA + "precios.csv", parse_dates=["Date"])
indice = pd.read_csv(RUTA + "indice.csv", parse_dates=["Date"])

# Compruebo el tamaño de cada tabla (filas y columnas)
print("Empresas:", empresas.shape)
print("Precios:", precios.shape)
print("Índice:", indice.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Empresas: (503, 4)
Precios: (1315666, 7)
Índice: (2708, 6)


Los tamaños coinciden con los que guardé al descargar los datos, así que los archivos están completos.

## 2. Tipos de datos

Miro qué tipo de dato tiene cada columna:

*Date* = Fecha (datetime64)

*Close/High/Low/Open* = Float (float64) ya que son precios e incluyen decimales

*Volume* = Entero (int64)

*Ticker* = Texto (object), en este caso hace referencia a las siglas de las empresas

In [5]:
# Muestro las columnas, su tipo de dato y cuántos valores tienen
precios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1315666 entries, 0 to 1315665
Data columns (total 7 columns):
 #   Column  Non-Null Count    Dtype         
---  ------  --------------    -----         
 0   Date    1315666 non-null  datetime64[ns]
 1   Close   1315666 non-null  float64       
 2   High    1315666 non-null  float64       
 3   Low     1315666 non-null  float64       
 4   Open    1315666 non-null  float64       
 5   Volume  1315666 non-null  int64         
 6   ticker  1315666 non-null  object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 70.3+ MB




Todas las columnas tienen el mismo número de valores que filas, así que en principio no falta ningún dato.

## 3. Valores nulos y duplicados

Compruebo si hay celdas vacías y si alguna empresa tiene el mismo día repetido.


In [6]:
# Cuento los valores vacíos de cada columna
# "isnull" me muestra cuantos valores vacios tiene cada columna.
print("Valores nulos por columna:")
print(precios.isnull().sum())

Valores nulos por columna:
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
ticker    0
dtype: int64


In [7]:
# Cuento las filas repetidas con la misma empresa y la misma fecha
duplicados = precios.duplicated(subset=["ticker", "Date"]).sum()
print("Filas duplicadas:", duplicados)

Filas duplicadas: 0


No hay ningún valor nulo ni ninguna fila duplicada.

## 4. Resumen estadístico

Uso la función "describe()" para ver el mínimo, el máximo, la media y otros valores de cada columna.
Busco sobre todo valores que no tengan sentido y me llamen la atención, como precios negativos.

In [8]:
# Muestro los números con 2 decimales en lugar de notación científica
pd.set_option("display.float_format", "{:,.2f}".format)

# Resumen estadístico
precios.describe()

,Date,Close,High,Low,Open,Volume
count,1315666,"1,315,666.00","1,315,666.00","1,315,666.00","1,315,666.00","1,315,666.00"
mean,2021-06-15 20:58:18.359005440,134.84,136.45,133.17,134.83,"6,617,770.67"
min,2016-01-04 00:00:00,0.61,0.62,0.60,0.60,0.00
25%,2018-10-11 00:00:00,42.14,42.64,41.64,42.14,"1,054,800.00"
50%,2021-06-29 00:00:00,79.07,79.98,78.15,79.07,"2,259,000.00"
75%,2024-02-26 00:00:00,150.17,151.94,148.35,150.17,"5,300,675.00"
max,2026-10-09 00:00:00,"9,924.40","9,964.77","9,794.00","9,914.17","3,692,928,000.00"
std,NaN,281.67,285.14,278.25,281.68,"26,571,963.76"


Información que obtengo tras ver la tabla:

- **No hay precios negativos ni cero, por lo que los datos están bien.**
- En el momento en que se ejecuta el codigo, el **precio medio es de unos 133 dólares, pero la mediana es de 78**, casi la mitad. Esto pasa porque hay algunas acciones muy caras (la más cara tiene un valor cercano a los 10.000 dólares) que suben la media. Por eso no tiene sentido comparar precios entre empresas, y en el análisis voy a trabajar con rentabilidades en %.
- **Hay días con volumen 0**, es decir, días en los que no se negoció ninguna acción. En empresas tan grandes es raro, así que lo reviso a continuación.

## 5. ¿Cuántos días de datos tiene cada empresa?

No todas las empresas llevan cotizando desde 2016. Para comparar rentabilidades a 10 años, todas deberían tener los mismos años de datos.

In [9]:
# Cuento cuántas filas (días) tiene cada empresa
dias_por_empresa = precios.groupby("ticker").size()

print("Mínimo de días:", dias_por_empresa.min())
print("Máximo de días:", dias_por_empresa.max())

Mínimo de días: 7
Máximo de días: 2708


In [10]:
# Las 10 empresas con menos días de datos
dias_por_empresa.sort_values().head(10)

,0
ticker,
VYLR,7
HONA,82
FDXF,95
Q,240
SNDK,416
GEV,637
SOLV,638
RDDT,641
VLTO,757


Las empresas con menos datos son sobre todo "spin-offs", es decir, empresas que se han separado de otras más grandes hace poco (por ejemplo GE Vernova de General Electric o Kenvue de Johnson & Johnson). También hay alguna salida a bolsa reciente, como Reddit.

### Comparo dos posibilidades
Como no todas las empresas tienen los mismos años de datos, tengo que decidir cuáles uso en el análisis. Si comparo la rentabilidad de una empresa con 10 años de datos con otra que solo tiene 3, el resultado no sería justo.

Calculo cuántas empresas me quedarían con cada opción:
1. Solo **empresas con datos desde el primer día (4 de enero de 2016)**. Así todas tienen los mismos 10 años y las comparaciones son equitativas.
2. **Empresas con al menos 5 años de datos**. Así pierdo menos empresas, pero tendría que comparar en periodos de 5 años en lugar de 10, que es un aspecto muy relevante en este tipo de inversión (el tiempo).

In [11]:
# Primera fecha de cada empresa
# La función "groupby" agrupa los datos por las siglas de las empresas y min() selecciona la fecha más antigua de cada una
primera_fecha = precios.groupby("ticker")["Date"].min()

# Opción 1: empresas que empiezan el primer día de los datos
completas = (primera_fecha == "2016-01-04").sum()
print("Empresas con datos desde el primer día (2016):", completas)

Empresas con datos desde el primer día (2016): 464


In [12]:
# Opción 2: empresas que tienen al menos 5 años de datos
# Busco el último día que tengo en los datos
ultima_fecha = precios["Date"].max()

# Le resto 5 años para saber desde qué fecha tiene que tener datos una empresa.
# Uso la función "DateOffset" en lugar de restar días porque así tiene en cuenta los años bisiestos
fecha_limite = ultima_fecha - pd.DateOffset(years=5)

cinco_anios = (primera_fecha <= fecha_limite).sum()
print("Fecha límite:", fecha_limite.date())
print("Empresas con al menos 5 años de datos:", cinco_anios)

Fecha límite: 2021-10-09
Empresas con al menos 5 años de datos: 491



**Decisión: me quedo con la opción 1.**

El objetivo de este proyecto es responder a qué habría pasado invirtiendo hace 10 años, y para comparar estrategias y sectores de forma justa todas las empresas tienen que tener esos 10 años. Con esta opción pierdo 38 empresas de 503, que es solo un 7,5 %. Me parece poco, así que no creo que cambie mucho los resultados.

## 6. Días con volumen 0

En el resumen estadístico vi que hay días sin ninguna acción negociada. Miro cuántos son y a qué empresas afectan.

In [13]:
# Me quedo solo con las filas donde el volumen es 0 igualando la columna [Volume] a 0
volumen_cero = precios[precios["Volume"] == 0]

print("Filas con volumen 0:", len(volumen_cero))

Filas con volumen 0: 2742


In [14]:
# Cuento cuántos días con volumen 0 tiene cada empresa
# La funcion "value_counts()" cuenta cuántas veces aparece cada valor en una columnas, "head(10)" muestra las 10 primeras y "rename()" cambia el titulo de la columna
volumen_cero["ticker"].value_counts().head(10).rename("Las 10 empresas con más días de volumen 0")


,Las 10 empresas con más días de volumen 0
ticker,
SW,1172
FERG,838
AMCR,669
HWM,35
VRT,12
IBKR,2
PFG,2
CNC,1
BKR,1


El volumen 0 no está repartido entre muchas empresas, sino que casi todo se concentra en tres: SW, FERG y AMCR. Entre las tres suman 2.679 de las 2.742 filas.
Si fueran uno o dos días sueltos en muchas empresas, serían fallos puntuales y bastaría con borrar esas filas. Pero al estar concentrado en tres empresas, parece que esas empresas en concreto tienen un problema en sus datos.

Tengo que investigar sus filas y sus fechas con más detalle.

In [15]:
# Miro algunas de esas filas
volumen_cero.head(15)

,Date,Close,High,Low,Open,Volume,ticker
61040,2016-01-04,29.19,29.19,29.19,29.19,0,AMCR
61042,2016-01-06,28.56,28.56,28.56,28.56,0,AMCR
61043,2016-01-07,28.56,28.56,28.56,28.56,0,AMCR
61044,2016-01-08,28.56,28.56,28.56,28.56,0,AMCR
61047,2016-01-13,26.32,26.32,26.32,26.32,0,AMCR
61048,2016-01-14,26.32,26.32,26.32,26.32,0,AMCR
61049,2016-01-15,26.32,26.32,26.32,26.32,0,AMCR
61050,2016-01-19,26.32,26.32,26.32,26.32,0,AMCR
61051,2016-01-20,26.32,26.32,26.32,26.32,0,AMCR
61052,2016-01-21,26.32,26.32,26.32,26.32,0,AMCR


Casi todos los casos están en tres empresas: **SW, FERG y AMCR**.

En las filas se ve que el precio de apertura, máximo, mínimo y cierre es el mismo, y que se repite varios días seguidos. En un mercado real el precio cambia todos los días, aunque sea un poco. Por eso creo que estos días no tienen datos reales, sino que se ha repetido el último precio para rellenar los huecos.

Para comprobarlo, miro en qué fechas empiezan y terminan los días con volumen 0 de estas empresas.

### Reviso las empresas sospechosas

Miro en qué fechas empiezan y terminan sus días con volumen 0.

In [16]:
# Lista con las empresas (Ticker = Siglas de empresas) que quiero revisar
sospechosas = ["SW", "FERG", "AMCR", "HWM", "VRT"]

# Recorro las empresas de la lista una a una
for ticker in sospechosas:
    # Filas con volumen 0 de esta empresa
    dias_cero = volumen_cero[volumen_cero["ticker"] == ticker]

    # Busco el primer y el último día con volumen 0
    inicio = dias_cero["Date"].min().date()
    fin = dias_cero["Date"].max().date()

    print(ticker, "| primera fecha:", primera_fecha[ticker].date(), "| volumen 0 desde", inicio, "hasta", fin)

SW | primera fecha: 2016-01-04 | volumen 0 desde 2016-01-05 hasta 2024-07-05
FERG | primera fecha: 2016-01-04 | volumen 0 desde 2016-01-04 hasta 2021-03-05
AMCR | primera fecha: 2016-01-04 | volumen 0 desde 2016-01-04 hasta 2019-06-10
HWM | primera fecha: 2016-11-01 | volumen 0 desde 2016-11-01 hasta 2016-12-22
VRT | primera fecha: 2018-08-02 | volumen 0 desde 2018-08-03 hasta 2019-11-29


**Conclusión:**

- SW, FERG y AMCR tienen volumen 0 durante un periodo seguido, desde 2016 hasta que empezaron a cotizar en la bolsa de EE. UU. (AMCR en 2019, FERG en 2021 y SW en 2024).
Antes cotizaban en otros países o eran otra empresa, así que los precios de esos años no son reales. Aunque su primera fecha sea 2016, son datos falsos que no puedo usar, así que las quito del análisis.

- HWM y VRT no me afectan, porque empiezan después de 2016 y ya estaban fuera de las 465 empresas que elegí.

- El resto de empresas solo tienen uno o dos días sueltos con volumen 0. Parecen fallos puntuales, así que borraré esas filas pero mantendré las empresas.



In [17]:
# Me quedo con las empresas que tienen datos desde el primer día
empresas_desde_2016 = primera_fecha[primera_fecha == "2016-01-04"].index

# Lista de empresas que quito por tener datos de relleno
excluidas = ["SW", "FERG", "AMCR"]

# Recorro las empresas desde 2016 y me quedo con las que no están en la lista de excluidas
# Se lee así: "para cada ticker t de empresas_desde_2016, guárdalo si no está en excluidas"
empresas_finales = [t for t in empresas_desde_2016 if t not in excluidas]

# Cuento cuántas empresas me quedan
print("Empresas que usaré en el análisis:", len(empresas_finales))

Empresas que usaré en el análisis: 461


Con esto me quedan 461 empresas para el análisis

## 7. Resumen de la exploración

### Calidad de los datos.

En general los datos están bien. No hay nulos ni duplicados, los tipos de datos son correctos y no hay precios negativos ni cero.

### Empresas que voy a utilizar.

Al final me quedo con 461 empresas. He quitado 39 porque empezaron a cotizar después de 2016, y así todas tienen los mismos años de datos. También he quitado SW, FERG y AMCR, porque aunque sus datos empiezan en 2016, antes de cotizar en EE. UU. los precios eran de relleno.

### Qué haré en la limpieza.

En el siguiente notebook voy a:

Quedarme solo con las 461 empresas.
Borrar las filas sueltas con volumen 0.
Quitar el último día si lo descargué con la bolsa abierta.
Poner los sectores en español.
Dejar la fecha sin la hora.
Guardar los datos limpios en Data/Processed.

### Cosas a tener en cuenta.

Uso las empresas que están ahora en el S&P 500, no las que había en 2016. Las que salieron del índice porque les fue mal no están, así que los resultados pueden salir un poco mejores de lo que fueron en realidad (esto se llama sesgo de supervivencia).
